# Tayseer Services Dashboard: Where Should SAR 40M Go?
Run all cells, then upload `tayseer_services.csv` when asked.

In [ ]:
!pip install -q dash

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

CHANNELS = ["Branch", "Call Center", "Mobile App", "Web Portal"]
BENCH = ["Riyadh", "Eastern Province"]
ORANGE, GREY, BLUE, SKY = "#D55E00", "#B0B0B0", "#0072B2", "#56B4E9"

df = pd.read_csv("tayseer_services.csv")
if pd.api.types.is_numeric_dtype(df["month"]):
    df["month"] = pd.to_datetime(df["month"], unit="D", origin="1899-12-30")
else:
    df["month"] = pd.to_datetime(df["month"])
df["year"] = df["month"].dt.year
MONTHS = sorted(df["month"].unique())

def wmean(d, col, by):
    d = d.assign(w=d[col] * d["transactions"]).groupby(by)[["w", "transactions"]].sum()
    return d["w"] / d["transactions"]

In [ ]:
def avg(d, col):
    return (d[col] * d["transactions"]).sum() / d["transactions"].sum()


def full_year(d):
    n = d.groupby("year")["month"].nunique()
    return n[n == 12].index.max() if (n == 12).any() else n.index.max()


def extra_hours(d):
    b = d[(d["channel"] == "Branch") & (d["year"] == full_year(d))]
    bench = wmean(b[b["region"].isin(BENCH)], "avg_completion_min", "service_category")
    gap = (b["avg_completion_min"] - b["service_category"].map(bench)).clip(lower=0)
    return (gap * b["transactions"] / 60).groupby(b["region"]).sum().sort_values()


EXTRA = extra_hours(df)
TOP3 = EXTRA.index[-3:][::-1]
SPLIT = (EXTRA[TOP3] / EXTRA[TOP3].sum() * 40).round()


def layout(fig, title, ytitle):
    fig.update_layout(title=title, template="simple_white", height=420, yaxis_title=ytitle)
    return fig


def trend_chart(d):
    t = wmean(d, "avg_completion_min", ["month", "channel"]).unstack()
    fig = go.Figure()
    for c in t.columns:
        fig.add_scatter(x=t.index, y=t[c], name=c, line=dict(color=ORANGE if c == "Branch" else GREY, width=3 if c == "Branch" else 1.5))
    fig.add_scatter(x=t.index, y=wmean(d, "avg_completion_min", "month"), name="All channels", line=dict(color="black", width=4))
    fig.update_yaxes(rangemode="tozero")
    return layout(fig, "Overall time fell a lot, but Branch time barely moved", "Minutes")


def region_chart(d):
    e = extra_hours(d) / 1000
    fig = go.Figure(go.Bar(x=e, y=e.index, orientation="h", text=e.round(), textposition="outside",
                           marker_color=[ORANGE if r in e.index[-3:] else GREY for r in e.index]))
    return layout(fig, "Three regions hold most of the extra branch time", "Region").update_xaxes(title="Thousand customer-hours")


def mix_chart(d):
    y0, y1 = d["year"].min(), full_year(d)
    share = {y: d[d["year"] == y].groupby("channel")["transactions"].sum() / d[d["year"] == y]["transactions"].sum() for y in (y0, y1)}
    time = {y: wmean(d[d["year"] == y], "avg_completion_min", "channel") for y in (y0, y1)}
    vals = [(share[y0] * time[y0]).sum(), (share[y0] * time[y1]).sum(), (share[y1] * time[y1]).sum()]
    fig = go.Figure(go.Bar(x=[f"Actual {y0}", f"{y1} with {y0} channel mix", f"Actual {y1}"], y=vals,
                           marker_color=[BLUE, SKY, BLUE], text=[f"{v:.1f}" for v in vals], textposition="outside"))
    return layout(fig, "Most of the drop is channel mix, not faster service", "Minutes").update_yaxes(range=[0, max(vals) * 1.2])


def budget_chart():
    fig = go.Figure(go.Bar(x=SPLIT.index, y=SPLIT, marker_color=ORANGE, text=[f"SAR {v:.0f}M" for v in SPLIT], textposition="outside"))
    return layout(fig, "Recommended split of SAR 40M", "SAR million").update_yaxes(range=[0, SPLIT.max() * 1.2])


def kpi(label, id):
    return html.Div([html.Small(label), html.H3(id=id)], style={"flex": 1, "border": "1px solid #ddd", "padding": "8px"})


def row(*cells, tag=html.Td):
    return html.Tr([tag(c, style={"border": "1px solid #ddd", "padding": "6px"}) for c in cells])

In [ ]:
app = Dash(__name__)
app.layout = html.Div(style={"fontFamily": "Arial", "maxWidth": "1100px", "margin": "auto"}, children=[
    html.H2("Where should SAR 40M go?"),
    html.P(html.B(f"Put the SAR 40M into branches in {TOP3[0]} (SAR {SPLIT.iloc[0]:.0f}M), {TOP3[1]} (SAR {SPLIT.iloc[1]:.0f}M) "
                  f"and {TOP3[2]} (SAR {SPLIT.iloc[2]:.0f}M). Start with Justice & Notary and Business & Licensing.")),
    html.Div([dcc.Dropdown(sorted(df["region"].unique()), multi=True, id="region", placeholder="Region"),
              dcc.Dropdown(sorted(df["service_category"].unique()), multi=True, id="category", placeholder="Service category"),
              dcc.Dropdown(CHANNELS, CHANNELS, multi=True, id="channel")]),
    dcc.RangeSlider(0, len(MONTHS) - 1, 1, value=[0, len(MONTHS) - 1], id="dates",
                    marks={i: str(pd.Timestamp(m).year) for i, m in enumerate(MONTHS) if pd.Timestamp(m).month == 1}),
    html.P("Average completion time fell from 25.9 min (2021) to 14.7 min (2026). Branch time only fell from 40.2 to 34.9 min."),
    html.Div([kpi("Completion time (min)", "k1"), kpi("CSAT (1-5)", "k2"), kpi("Cost per transaction (SAR)", "k3"),
              kpi("Complaints per 10k", "k4"), kpi("Digital adoption (%)", "k5")], style={"display": "flex", "gap": "8px"}),
    dcc.Graph(id="trend"),
    html.P(f"{', '.join(TOP3)} hold {EXTRA[TOP3].sum() / EXTRA.sum():.0%} of the extra branch time compared with Riyadh and Eastern Province for the same services."),
    dcc.Graph(id="regions"),
    html.P("If the 2021 channel mix had stayed, 2025 would be 23.0 min, not 15.8. So 71% of the drop is people moving to the app. "
           "In the three regions, Justice & Notary and Business & Licensing take about 120 min per branch visit, vs about 45 min in Riyadh."),
    dcc.Graph(id="mix"),
    html.Table([row("Option", "Result", tag=html.Th),
                row("A. Spread across all branches", "Only SAR 16M reaches the three slow regions"),
                row("B. Spend it all on digital", "Does not help the 31.7M transactions still done in branches"),
                row("C. Focus on the three regions", "Covers 79% of the extra branch time")], style={"borderCollapse": "collapse", "width": "100%"}),
    html.P("Option C is best. If these branches reach Riyadh/Eastern times, people could save up to 3.1M hours a year. This is a what-if, not a forecast."),
    dcc.Graph(figure=budget_chart()),
    html.P(html.B("The ask: approve SAR 40M for these branches and track branch time by region every quarter.")),
])

In [ ]:
@app.callback([Output(f"k{i}", "children") for i in range(1, 6)] + [Output("trend", "figure"), Output("regions", "figure"), Output("mix", "figure")],
              [Input("region", "value"), Input("category", "value"), Input("channel", "value"), Input("dates", "value")])
def update(regions, categories, channels, dates):
    d = df[df["month"].between(MONTHS[dates[0]], MONTHS[dates[1]])]
    if categories:
        d = d[d["service_category"].isin(categories)]
    all_regions = d
    if regions:
        d = d[d["region"].isin(regions)]
    adoption = d.drop_duplicates(["month", "region", "service_category"])["digital_adoption_pct"].mean()
    d = d[d["channel"].isin(channels or CHANNELS)]
    kpis = [f"{avg(d, 'avg_completion_min'):.1f}", f"{avg(d, 'csat'):.2f}", f"{avg(d, 'cost_per_txn_sar'):.2f}",
            f"{d['complaints'].sum() / d['transactions'].sum() * 10000:.1f}", f"{adoption:.1f}"]
    return kpis + [trend_chart(d), region_chart(all_regions), mix_chart(d)]

In [ ]:
app.run(jupyter_mode="inline")